# Dashboard de Configuración

In [ ]:
# Reporte

NOMBRE_REPORTE = "REPORTE 1 - CAFETERIAS"
CARPETA_OUTPUT = "prueba" # Cambiar nombre para que no se sustituya la información del reporte actual

In [ ]:
CODIGO_DENUE = "722515"

TIPO_DE_DENUE = "Cafeterias"  # Código para cafeterías y restaurantes que sirven bebidas.

# Para la consulta de códigos utilicese scian_2023_categorias_y_productos.xlsx que se
# encuentra en este mismo directorio dentro de la carpeta de bases.

PALABRAS_CLAVE = [
        'caf', 'coffee', 'café', 'cafe', 'espresso', 'latte', 'ristretto',
        'capuccino', 'cappuccino', 'mocha', 'starbuck', 'barista', 'coffeebar',
        'coffee shop', 'caffè', 'caffe'
    ]

# Si es necesario cambiar las palabras clave, en vez de hacerlo manual, utilicen
# este prompt con Gemini que está incluido en Colab:
# "Modifica la variable PALABRAS_CLAVE para que contenga strings relacionados con
# [AREA NUEVA DE DENUE] en vez de cafeterías. Es decir, palabras clave (singulares,
# no compuestas) que puedan usarse para identificar negocios con relación."

In [ ]:
# Para utilizar el programa hay que crear una cuenta gratuita de OPENROUTESERVICE.
# Una vez se tenga la cuenta, pegar la llave obtenida del dashboard: https://account.heigit.org/manage/key

API_KEY = ""

In [ ]:
# Parámetros de isócronas.

DRIVING_MINUTES = 5
WALKING_MINUTES = 5


# Para calcular nuevas coordenadas, utilice Coordenadas.ipynb en el mismo directorio
# que este cuaderno. Si se quieren obtener coordenadas de una dirección, utilicen
# Google Maps para obtenerlas de forma manual.

loc_Qro = (-100.3831, 20.5888) # Ubicación del mapa en un punto
locations = [
    (-100.3831, 20.5888)
]

location_names = [
    "PUNTO 1"
]

loc_Qro = (-100.3831, 20.5888) #Ubicación del mapa en un punto
locations = [
    (-100.3636904550087, 20.560252144338826),
    (-100.44696444611094, 20.70773033593715),
    (-100.38774838766155, 20.611873073029546),
    (-100.43266189263078, 20.55978792766156)
]

location_names = [
    "CENTRO SUR",
    "JURIQUILLA",
    "LA LABORCILLA",
    "EL JACAL"
]

In [ ]:
# Rutas a las bases de datos.

from google.colab import drive
import os

!rm -rf /content/sample_data

if not os.path.exists('/content/databases'):
    os.makedirs('/content/databases')

drive.mount('/content/databases/drive') #Nombre de la carpeta

BASE_DIR = "/content/databases/drive/MyDrive/Análisis de Servicios"

MANZANAS_SHP = os.path.join(BASE_DIR, "bases/22_Manzanas_INV2020_shp.zip")
ENIGH_CSV    = os.path.join(BASE_DIR, "bases/conjunto_de_datos_concentradohogar_enigh2022_ns.csv")
DENUE_ZIP    = os.path.join(BASE_DIR, "bases/denue_22_csv.zip")
LOGO_PNG     = os.path.join(BASE_DIR, "bases/logo_aristor_01.png")

# Ejecución de Algoritmo

In [ ]:
#@title Cálculo de Isócronas y Población

!pip install -q openrouteservice selenium reportlab

import zipfile
import warnings
warnings.filterwarnings("ignore")

import sys
import io
import pandas as pd
import geopandas as gpd
import openrouteservice
import folium
from folium.plugins import MarkerCluster
from folium import Icon, DivIcon
import branca
from shapely.geometry import Point, Polygon, MultiPolygon
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import re
import numpy as np
from PIL import Image

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader
from datetime import datetime
from reportlab.platypus import Table, TableStyle
from reportlab.lib import colors
import shutil


# Creamos un string buffer para capturar output de consola
original_stdout = sys.stdout  # Referencia original del stream

captured_output = io.StringIO()
sys.stdout = captured_output  # Redirige salida


# ---------------- 0) CARGA DE VARIABLES  ----------------

try:
    manzanas
    print("Variable 'manzanas' ya en memoria.")
except NameError:
    print("Cargando manzanas...")
    with zipfile.ZipFile(MANZANAS_SHP, "r") as z:
        z.extractall("/content/databases/manzanas")
    manzanas = gpd.read_file(
        "/content/databases/manzanas/22_Manzanas_INV2020_shp/INV2020_IND_PVEU_MZA_22.shp"  # Ojo. Si se cambia la base de datos hay que actualizar este directorio
    ).to_crs(epsg=4326)
    print("Manzanas:", len(manzanas))

try:
    enigh
    print("Variable 'enigh' ya en memoria.")
except NameError:
    print("Cargando ENIGH...")
    enigh = pd.read_csv(ENIGH_CSV, encoding="latin-1", low_memory=False)
    print("ENIGH filas:", len(enigh))

try:
    gdf_denue
    print("Base gdf_denue ya en memoria.")
except NameError:
    if os.path.exists(DENUE_ZIP):
        with zipfile.ZipFile(DENUE_ZIP, "r") as z:
            z.extractall("/content/databases/denue_qro")
        denue = pd.read_csv("/content/databases/denue_qro/conjunto_de_datos/denue_inegi_22_.csv",  # Nuevamente, actualizar directorio si hay cambio de base de datos
                            encoding="latin-1", low_memory=False, on_bad_lines="skip")
        denue = denue.dropna(subset=["latitud","longitud"])
        denue["latitud"] = pd.to_numeric(denue["latitud"], errors="coerce")
        denue["longitud"] = pd.to_numeric(denue["longitud"], errors="coerce")
        denue = denue.dropna(subset=["latitud","longitud"])

        denue_services = denue[denue['codigo_act'].astype(str).str.startswith(CODIGO_DENUE[:3])].copy()
        gdf_denue = gpd.GeoDataFrame(denue_services,
                                     geometry=[Point(xy) for xy in zip(denue_services.longitud,
                                                                       denue_services.latitud)],
                                     crs="EPSG:4326")
        print(f"DENUE {TIPO_DE_DENUE} cargados con {CODIGO_DENUE}:", len(gdf_denue))
    else:
        print("No se encontró DENUE; continuamos sin capa de establecimientos.")


def filtrar_servicios(gdf_denue):
    """
    Devuelve gdf_services: solo los establecimientos detectados como servicios.
    Usa el código del servicio especificado en CODIGO_DENUE además de filtros.
    """
    services_codes = [CODIGO_DENUE]  # código principal para servicios
    keywords = PALABRAS_CLAVE.copy()
    pattern = re.compile(r'(' + r'|'.join(re.escape(k) for k in keywords) + r')', flags=re.IGNORECASE)

    # máscara por código (string) y por nombre (nom_estab o nombre_act)
    mask_code = gdf_denue['codigo_act'].astype(str).isin(services_codes)
    nom_estab = gdf_denue['nom_estab'].astype(str).fillna('')
    nombre_act = gdf_denue.get('nombre_act', pd.Series('')).astype(str).fillna('')
    mask_name = nom_estab.str.contains(pattern) | nombre_act.str.contains(pattern)

    # combinar (usamos OR para no perder servicios mal codificados)
    mask_services = mask_code | mask_name
    gdf_services = gdf_denue[mask_services].copy()

    # normalizar coordenadas numéricas
    gdf_services['latitud'] = pd.to_numeric(gdf_services['latitud'], errors='coerce')
    gdf_services['longitud'] = pd.to_numeric(gdf_services['longitud'], errors='coerce')
    gdf_services = gdf_services.dropna(subset=['latitud','longitud']).copy()

    return gdf_services

gdf_services = filtrar_servicios(gdf_denue)
print(f"Se detectaron {len(gdf_services)} {TIPO_DE_DENUE}.")

if not os.path.exists('/content/output'):
    os.makedirs('/content/output')

if not os.path.exists('/content/output/mapas'):
    os.makedirs('/content/output/mapas')

if not os.path.exists('/content/databases/resources'):
    os.makedirs('/content/databases/resources')

output_path = os.path.join(BASE_DIR, f"reportes/{CARPETA_OUTPUT}")

if not os.path.exists(output_path):
    os.makedirs(output_path)


maps_path = os.path.join(output_path, f"mapas")

if not os.path.exists(maps_path):
    os.makedirs(maps_path)

resources_path = os.path.join(output_path, f"recursos")

if not os.path.exists(resources_path):
    os.makedirs(resources_path)


# ---------------- 1) NORMALIZAR CLAVES EN MANZANAS ----------------
for col, width in [('CVE_ENT',2), ('CVE_MUN',3), ('CVE_LOC',4), ('CVE_AGEB',4), ('CVE_MZA',3)]:
    if col in manzanas.columns:
        manzanas[col] = manzanas[col].astype(str).str.zfill(width)
    else:
        manzanas[col] = ""

manzanas['CVE_AGEB_4'] = manzanas['CVE_AGEB'].astype(str).str.zfill(4)


# ---------------- 2) CONVERTIR POBTOT A NUMÉRICO Y AGRUPAR POR AGEB ----------------

# Si existe POBTOT a nivel de manzana, convertirlo; si no, creamos columna con NaN

if 'POBTOT' in manzanas.columns:
    manzanas['POBTOT'] = pd.to_numeric(manzanas['POBTOT'], errors='coerce')
else:
    manzanas['POBTOT'] = pd.NA


# Disolver manzanas por AGEB y sumar población correctamente
agebs = manzanas.dissolve(by='CVE_AGEB_4', aggfunc={'POBTOT':'sum'}).reset_index()
agebs = gpd.GeoDataFrame(agebs, geometry='geometry', crs="EPSG:4326")
agebs.rename(columns={'POBTOT':'POBTOT_sum'}, inplace=True)
agebs['POBTOT_sum'] = pd.to_numeric(agebs['POBTOT_sum'], errors='coerce').fillna(0).astype(float)


# ---------------- 3) PREPARAR ENIGH: EXTRAER AGEB Y CALCULAR est_socio_mean ----------------

enigh['ubica_geo'] = enigh['ubica_geo'].astype(str).str.strip()
def extract_ageb_code(s):
    s = str(s).strip()
    if len(s) >= 13:
        return s[-4:]
    else:
        return s.zfill(4)[-4:]
enigh['AGEB_code'] = enigh['ubica_geo'].apply(extract_ageb_code)

# Agregación ponderada
enigh_valid = enigh.dropna(subset=['est_socio','factor','AGEB_code']).copy()
enigh_valid['est_socio'] = pd.to_numeric(enigh_valid['est_socio'], errors='coerce')
enigh_valid['factor'] = pd.to_numeric(enigh_valid['factor'], errors='coerce')
enigh_valid = enigh_valid.dropna(subset=['est_socio','factor'])
enigh_valid['wt_est'] = enigh_valid['est_socio'] * enigh_valid['factor']

enigh_ageb = enigh_valid.groupby('AGEB_code', as_index=False).agg(
    est_socio_wsum = ('wt_est','sum'),
    factor_sum = ('factor','sum'),
    hogares = ('foliohog','count')
)
enigh_ageb['est_socio_mean'] = enigh_ageb['est_socio_wsum'] / enigh_ageb['factor_sum']
enigh_ageb['AGEB_code'] = enigh_ageb['AGEB_code'].astype(str).str.zfill(4)

# ---------------- 4) MERGE LIMPIO DE ENIGH->AGEB ----------------

# Eliminar columnas conflictivas en agebs
for c in ['AGEB_code','est_socio_mean','factor_sum','hogares']:
    if c in agebs.columns:
        agebs.drop(columns=c, inplace=True, errors='ignore')

agebs = agebs.merge(
    enigh_ageb[['AGEB_code','est_socio_mean','factor_sum','hogares']],
    left_on='CVE_AGEB_4',
    right_on='AGEB_code',
    how='left'
)
if 'AGEB_code' in agebs.columns:
    agebs.drop(columns=['AGEB_code'], inplace=True)

print(f"AGEBs totales: {len(agebs)}. AGEBs sin est_socio_mean tras merge: {agebs['est_socio_mean'].isna().sum()}")


# ---------------- 5) IMPUTACIÓN ESPACIAL POR VECINO MÁS CERCANO (basado en centroide) ----------------

nonmissing = agebs[agebs['est_socio_mean'].notna()].copy()
missing = agebs[agebs['est_socio_mean'].isna()].copy()

if len(missing) > 0 and len(nonmissing) > 0:
    nonmissing_c = nonmissing.copy()
    missing_c = missing.copy()
    nonmissing_c['geometry'] = nonmissing_c.geometry.centroid
    missing_c['geometry'] = missing_c.geometry.centroid
    nonmissing_c = nonmissing_c[['CVE_AGEB_4','geometry','est_socio_mean']].rename(columns={'est_socio_mean':'est_socio_mean_nb'})
    nearest = gpd.sjoin_nearest(missing_c, nonmissing_c, how='left', distance_col='dist')
    # nearest retains the same index as missing_c (which is original agebs index)
    agebs.loc[nearest.index, 'est_socio_mean'] = nearest['est_socio_mean_nb'].values
    print("Imputación espacial realizada. AGEBs aún sin valor:", agebs['est_socio_mean'].isna().sum())
else:
    print("No se realizó imputación espacial (no hay faltantes o no hay AGEBs con valor).")

# Fallback global si aún faltan
if agebs['est_socio_mean'].isna().sum() > 0:
    global_mean = enigh_ageb['est_socio_mean'].mean()
    agebs['est_socio_mean'] = agebs['est_socio_mean'].fillna(global_mean)
    print("Se rellenaron AGEBs restantes con promedio global:", round(global_mean,3))


# ---------------- 6) PROPAGAR est_socio_mean A NIVEL DE MANZANA ----------------

# Eliminar col conflictiva y hacer merge (manzanas conserva geometría)
if 'est_socio_mean' in manzanas.columns:
    manzanas.drop(columns=['est_socio_mean'], inplace=True, errors='ignore')

manzanas = manzanas.merge(
    agebs[['CVE_AGEB_4','est_socio_mean']],
    on='CVE_AGEB_4',
    how='left'
)

# Asegurar numerico
manzanas['est_socio_mean'] = pd.to_numeric(manzanas['est_socio_mean'], errors='coerce')


# ---------------- 6.5) calcular proporciones por clase usando est_socio_mean ya propagado a manzanas ----------

def socio_class_props_from_manzanas(iso_gdf, manzanas_gdf):
    """
    Usa est_socio_mean en las manzanas (ya imputado) para calcular counts y proporciones de class_1..class_4.
    - No re-imputa nada.
    - Reparte fraccionalmente entre las dos clases adyacentes (ej 1.7 -> 0.3 clase1, 0.7 clase2).
    - Peso por manzana: POBTOT si existe y >0; si no, 1.
    Devuelve: counts (dict), props (dict), total_weight (float)
    """

    geom = iso_gdf.loc[0, 'geometry']
    manz_inside = manzanas_gdf[manzanas_gdf.intersects(geom)].copy()

    cols = ['class_1','class_2','class_3','class_4']
    counts = {c: 0.0 for c in cols}

    if manz_inside.empty:
        props = {c: 0.0 for c in cols}
        return counts, props, 0.0

    # asegurar POBTOT disponible
    if 'POBTOT' not in manz_inside.columns:
        manz_inside['POBTOT'] = 0.0
    manz_inside['POBTOT'] = pd.to_numeric(manz_inside['POBTOT'], errors='coerce').fillna(0.0)

    for _, row in manz_inside.iterrows():
        est = row.get('est_socio_mean', None)
        if est is None or pd.isna(est):
            # NO re-imputar: saltar manzanas sin est_socio_mean
            continue
        # limitar a [1,4]
        est = float(min(max(est, 1.0), 4.0))
        lower = int(np.floor(est))
        upper = int(np.ceil(est))

        if lower == upper:
            w_lower = 1.0
            w_upper = 0.0
        else:
            w_upper = est - lower
            w_lower = 1.0 - w_upper

        # peso: preferir POBTOT (si >0), si no usar 1.0
        peso = float(row['POBTOT']) if row['POBTOT'] and row['POBTOT'] > 0 else 1.0

        counts[f'class_{lower}'] += peso * w_lower
        counts[f'class_{upper}'] += peso * w_upper

    total = sum(counts.values())
    props = {c: (counts[c] / total if total > 0 else 0.0) for c in cols}
    return counts, props, total


# ---------------- 7) INICIALIZACIÓN DE OpenRouteService ----------------

client = openrouteservice.Client(key=API_KEY)

def build_iso(coords, profile, seconds):
    iso = client.isochrones(locations=[coords], profile=profile, range=[seconds])
    iso_gdf = gpd.GeoDataFrame.from_features(iso['features'])
    iso_gdf.set_crs(epsg=4326, inplace=True)
    return iso_gdf

# ---------------- 8) CALCULAR INFO DENTRO DE ISÓCRONA ----------------

def display_info_from_iso(iso_gdf):
    geom = iso_gdf.loc[0,'geometry']
    agebs_inside = agebs[agebs.intersects(geom)]
    # establecimientos
    num_estab = None
    if 'gdf_denue' in globals():
        estab_inside = gdf_denue[gdf_denue.intersects(geom)]
        num_estab = len(estab_inside)
    # poblacion total (usar POBTOT_sum numérico)
    pop_total = int(agebs_inside['POBTOT_sum'].fillna(0).sum()) if 'POBTOT_sum' in agebs_inside.columns else None
    # socio promedio (ponderado por factor_sum si existe)
    socio_avg = None
    if 'factor_sum' in agebs_inside.columns and agebs_inside['factor_sum'].notna().sum() > 0:
        valid = agebs_inside.dropna(subset=['est_socio_mean','factor_sum'])
        if len(valid)>0 and valid['factor_sum'].sum()>0:
            socio_avg = (valid['est_socio_mean'] * valid['factor_sum']).sum() / valid['factor_sum'].sum()
            socio_avg = round(float(socio_avg),3)
    else:
        v = agebs_inside['est_socio_mean'].dropna()
        if len(v)>0:
            socio_avg = round(float(v.mean()),3)
    return len(agebs_inside), num_estab, pop_total, socio_avg

# ---------------- 9) MAPA: PARA CADA ISÓCRONA, COLOREAR SOLO LAS MANZANAS DENTRO (capas independientes) ----------------

def add_services_cluster(map_obj, geom, gdf_services, name=CODIGO_DENUE):
    """
    Añade un MarkerCluster de servicios que intersectan 'geom' al folium.Map map_obj.
    Devuelve el número de servicios añadidos.
    """
    # asegurar geometría válida
    services_inside = gdf_services[gdf_services.intersects(geom)].copy()
    if services_inside.empty:
        return 0
    mc = MarkerCluster(name=name, overlay=True, control=True)
    for _, r in services_inside.iterrows():
        lat = float(r['latitud']); lon = float(r['longitud'])
        popup_html = f"<b>{r.get('nom_estab','Sin nombre')}</b><br>Act: {r.get('codigo_act','')}<br>{r.get('nombre_act','')}"
        icon = Icon(icon='star', prefix='fa', color='darkred')
        folium.Marker([lat, lon], popup=popup_html, icon=icon).add_to(mc)
    mc.add_to(map_obj)
    return len(services_inside)

def build_map_iso_heats(center, iso_gdfs, names, manzanas_gdf, origins=None):
    m = folium.Map(location=[center[1], center[0]], zoom_start=13)
    vals = manzanas_gdf['est_socio_mean'].dropna()
    if len(vals) > 0:
        vmin, vmax = float(vals.min()), float(vals.max())
    else:
        vmin, vmax = 0.0, 1.0
    colormap = branca.colormap.LinearColormap(
        ['blue','cyan','yellow','orange','red'], vmin=vmin, vmax=vmax
    )
    colormap.caption = "Nivel socioeconómico (est_socio_mean)"
    colormap.add_to(m)

    for idx, (iso_gdf, name) in enumerate(zip(iso_gdfs, names), start=1):
        geom = iso_gdf.loc[0, 'geometry']
        # polígono
        folium.GeoJson(
            geom,
            name=f"{name} (isochrone)",
            style_function=lambda x: {'color':'black','weight':2,'fillOpacity':0.0}
        ).add_to(m)

        # Añadir capa de servicios filtradas (gdf_services)
        if 'gdf_services' in globals():
            added = add_services_cluster(m, geom, gdf_services, name=f"{TIPO_DE_DENUE} {name}")


        # marcador con número si tenemos origins
        if origins:
            lon, lat = origins[idx-1]
            folium.Marker(
                location=[lat, lon],
                icon=folium.DivIcon(html=f"""
                    <div style="font-size:14px; font-weight:bold; color:black;
                                background-color:white; border-radius:50%;
                                width:24px; height:24px; text-align:center;
                                line-height:24px; border:1px solid black;">
                        {idx}
                    </div>
                """)
            ).add_to(m)

        # manzanas dentro
        manz_inside = manzanas_gdf[manzanas_gdf.intersects(geom)].copy()
        if manz_inside.empty:
            continue
        def style_fn(feature):
            val = feature['properties'].get('est_socio_mean', None)
            if val is None or pd.isna(val):
                return {'fillOpacity':0.0, 'color':'#999999', 'weight':0.2}
            color = colormap(val)
            return {'fillColor': color, 'fillOpacity': 0.7, 'color': color, 'weight': 0.2}

        folium.GeoJson(
            data=manz_inside.to_json(),
            name=f"Manzanas {name}",
            style_function=style_fn,
            tooltip=folium.GeoJsonTooltip(
                fields=['CVE_AGEB_4','est_socio_mean','POBTOT'],
                aliases=['AGEB','est_socio','POBTOT_manzana'],
                localize=True
            )
        ).add_to(m)

    folium.LayerControl().add_to(m)
    return m

def count_services_in_iso(iso_gdf):
    geom = iso_gdf.loc[0, 'geometry']
    if 'gdf_services' in globals():
        return int(gdf_services.intersects(geom).sum())
    return 0  # si no existe gdf_services, devuelve 0


# ---------------- 10) EJECUTAR ISÓCRONAS ----------------

print(f"\n{WALKING_MINUTES} Minutos Caminando:")
print(f"{'        ':<12}{'AGEBs':>8}{TIPO_DE_DENUE:>15}{'Poblacion':>15}{'Est_Socio_Prom':>18}")

iso_gdfs_walk = []
iso_names_walk = []
for i, loc in enumerate(locations, start=1):
    try:
        iso_gdf = build_iso(loc, "foot-walking", WALKING_MINUTES*60)
        iso_gdfs_walk.append(iso_gdf)
        iso_names_walk.append(f"Walk {i}")

        agebs_cnt, _, pop_total, socio_avg = display_info_from_iso(iso_gdf)
        services_cnt = count_services_in_iso(iso_gdf)

        counts, props, total_w = socio_class_props_from_manzanas(iso_gdf, manzanas)

        prop_str = " / ".join([f"{props[f'class_{k}']*100:.1f} %" for k in [1,2,3,4]])
        print(f"{f'Ubicación {i}':<12}{agebs_cnt:>8}{services_cnt:>15}{(pop_total or 0):>15}{(socio_avg or 0):>18.2f}   {prop_str}")

    except Exception as e:
        print(f"{f'Ubicación {i}':<12}{'Error':>8}    {e}")

print(f"\n{DRIVING_MINUTES} Minutos en Coche:")
print(f"{'        ':<12}{'AGEBs':>8}{TIPO_DE_DENUE:>15}{'Poblacion':>15}{'Est_Socio_Prom':>18}")

iso_gdfs_drive = []
iso_names_drive = []
for i, loc in enumerate(locations, start=1):
    try:
        iso_gdf = build_iso(loc, "driving-car", DRIVING_MINUTES*60)
        iso_gdfs_drive.append(iso_gdf)
        iso_names_drive.append(f"Drive {i}")

        agebs_cnt, _, pop_total, socio_avg = display_info_from_iso(iso_gdf)
        services_cnt = count_services_in_iso(iso_gdf)

        counts, props, total_w = socio_class_props_from_manzanas(iso_gdf, manzanas)

        prop_str = " / ".join([f"{props[f'class_{k}']*100:.1f} %" for k in [1,2,3,4]])
        print(f"{f'Ubicación {i}':<12}{agebs_cnt:>8}{services_cnt:>15}{(pop_total or 0):>15}{(socio_avg or 0):>18.2f}   {prop_str}")

    except Exception as e:
        print(f"{f'Ubicación {i}':<12}{'Error':>8}    {e}")



# Restoramos el output normal
sys.stdout = original_stdout  # usamos la referencia guardada


# Accedemos al output capturado y lo mostramos en salida normal.
contenido = captured_output.getvalue()
print("Lo capturado fue:\n\n" + contenido)


In [ ]:
#@title Visualización Isócrona Walking

# ---------------- 11) CONSTRUCCIÓN MAPAS DE ISÓCRONA WALKING ----------------
mapa_walk = build_map_iso_heats(
    loc_Qro,
    iso_gdfs_walk,
    iso_names_walk,
    manzanas,
    origins=locations  # pasamos las coordenadas originales
)


mapa_walk.save('/content/output/mapas/mapa_walk.html')
mapa_walk.save(os.path.join(maps_path, 'mapa_walk.html'))
mapa_walk

In [ ]:
#@title Visualización Isócrona Driving

# ---------------- 12) CONSTRUCCIÓN MAPAS DE ISÓCRONA DRIVING ----------------
mapa_drive = build_map_iso_heats(
    loc_Qro,
    iso_gdfs_drive,
    iso_names_drive,
    manzanas,
    origins=locations
)
mapa_drive.save('/content/output/mapas/mapa_drive.html')
mapa_drive.save(os.path.join(maps_path, 'mapa_drive.html'))
mapa_drive

In [ ]:
#@title Visualización Isócrona Combinado

center = loc_Qro

# Creamos mapa base
mapa_combined = folium.Map(location=[center[1], center[0]], zoom_start=13)

# Creamos un colormap
vals = manzanas['est_socio_mean'].dropna()
if len(vals) > 0:
    vmin, vmax = float(vals.min()), float(vals.max())
else:
    vmin, vmax = 0.0, 1.0
colormap = branca.colormap.LinearColormap(['blue','cyan','yellow','orange','red'], vmin=vmin, vmax=vmax)
colormap.caption = "Nivel socioeconómico (est_socio_mean)"
colormap.add_to(mapa_combined)

# Anillos exteriores
def exterior_coords_list(geom):
    rings = []
    if isinstance(geom, Polygon):
        rings.append(list(geom.exterior.coords))
    elif isinstance(geom, MultiPolygon):
        for poly in geom.geoms:
            rings.append(list(poly.exterior.coords))
    else:
        try:
            poly = Polygon(geom)
            rings.append(list(poly.exterior.coords))
        except Exception:
            pass
    return rings

# style_fn global para manzanas (reusa colormap)
def style_fn(feature, cmap=colormap):
    try:
        val = feature['properties'].get('est_socio_mean', None)
        if val is None or pd.isna(val):
            return {'fillOpacity':0.0, 'color':'#999999', 'weight':0.2}
        color = cmap(float(val))
        return {'fillColor': color, 'fillOpacity': 0.7, 'color': color, 'weight': 0.2}
    except Exception:
        return {'fillOpacity':0.0, 'color':'#999999', 'weight':0.2}

# Creamos un FeatureGroup para los labels (se añade al final para que este sobre todo)
labels_fg = folium.FeatureGroup(name="Isócrona labels", show=True)

# --- Para cada isócrona de DRIVING: dibujar su polígono, pintar manzanas dentro con heat y añadir servicios ---
for i, iso in enumerate(iso_gdfs_drive, start=1):
    geom_drive = iso.loc[0, 'geometry']  # shapely geometry

    # 1) Dibujar el contorno del driving (outline)
    folium.GeoJson(
        data = geom_drive.__geo_interface__,
        name = f"Driving {i} (outline)",
        style_function = lambda feat: {'color':'black', 'weight':2, 'fillOpacity':0.0}
    ).add_to(mapa_combined)

    # 2) Seleccionar manzanas que intersecten la isócrona driving
    manz_inside = manzanas[manzanas.intersects(geom_drive)].copy()

    # Si hay manzanas, pintarlas como heat
    if not manz_inside.empty:
        folium.GeoJson(
            data = manz_inside.to_json(),
            name = f"Manzanas Driving {i} (heat)",
            style_function = style_fn,
            tooltip = folium.GeoJsonTooltip(
                fields=['CVE_AGEB_4','est_socio_mean','POBTOT'],
                aliases=['AGEB','est_socio','POBTOT_manzana'],
                localize=True
            )
        ).add_to(mapa_combined)

    # 3) Añadir MarkerCluster con servicios dentro de esta isócrona (misma forma que en driving)
    if 'gdf_services' in globals():
        services_inside = gdf_services[gdf_services.intersects(geom_drive)].copy()
        if not services_inside.empty:
            mc = MarkerCluster(name=f"Cafeterías Driving {i}", overlay=True, control=True)
            for _, r in services_inside.iterrows():
                lat = float(r['latitud']); lon = float(r['longitud'])
                popup_html = f"<b>{r.get('nom_estab','Sin nombre')}</b><br>Act: {r.get('codigo_act','')}<br>{r.get('nombre_act','')}"
                icon = Icon(icon='star', prefix='fa', color='darkred')
                folium.Marker([lat, lon], popup=popup_html, icon=icon).add_to(mc)
            mc.add_to(mapa_combined)

    # 4) Agregar el marcador numerado (LABEL) al group labels_fg
    #    usamos las coordenadas originales (locations) si las tienes; si no, usamos centroid del geom_drive
    try:
        lon0, lat0 = locations[i-1]   # locations es tu lista de (lon, lat)
        label_loc = (lat0, lon0)
    except Exception:
        # fallback al centroid del polígono
        c = geom_drive.centroid
        label_loc = (c.y, c.x)

    div_html = f"""
        <div style="font-size:14px; font-weight:bold; color:black;
                    background-color:white; border-radius:50%;
                    width:28px; height:28px; text-align:center;
                    line-height:28px; border:1px solid black;">
            {i}
        </div>
    """
    # Añadimos al FeatureGroup (no al mapa directo)
    folium.Marker(
        location=label_loc,
        icon=DivIcon(html=div_html),

        # aseguramos que intente renderizar por encima con z_index_offset
        z_index_offset=1000
    ).add_to(labels_fg)

# --- Dibujamos outlines de WALKING (igual que antes) ---
for j, iso in enumerate(iso_gdfs_walk, start=1):
    geom_walk = iso.loc[0, 'geometry']
    rings = exterior_coords_list(geom_walk)
    if not rings:
        continue

    # Halo blanco grueso (para que destaque sobre el fill)
    for ring in rings:
        latlon = [(y, x) for x, y in ring]  # folium usa (lat, lon)
        folium.PolyLine(locations=latlon, color='white', weight=2, opacity=0.5,
                        dash_array='10,6', tooltip=f"Walking halo {j}").add_to(mapa_combined)

    # Línea roja fina encima
    for ring in rings:
        latlon = [(y, x) for x, y in ring]
        folium.PolyLine(locations=latlon, color='#d62728', weight=3, opacity=1.0,
                        dash_array='6,4', tooltip=f"Walking outline {j}").add_to(mapa_combined)


# --- Añadimos al FeatureGroup ---
labels_fg.add_to(mapa_combined)

# Control de capas y guardar
folium.LayerControl().add_to(mapa_combined)
mapa_combined.save('/content/output/mapas/mapa_combined.html')
mapa_combined.save(os.path.join(maps_path, 'mapa_combined.html'))


# Mostramos mapa combinado en notebook
mapa_combined

In [ ]:
#@title Análisis


# Función general
# --------------------------------------------- #
def generate_single_location_map(location, iso_gdf_walk, iso_gdf_drive, name, manzanas_gdf):
    """Generates a Folium map for a single location and its isochrone."""
    lon, lat = location
    center = (lon, lat) # Centro del mapa en coordenadas.

    m = folium.Map(location=[lat, lon], zoom_start=14) # Aumentamos zoom

    # Obtenemos los límites de la geometría exterior
    bounds = iso_gdf_drive.total_bounds  # [minx, miny, maxx, maxy]

    # Ajustamos el mapa a esos límites
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

    # Añadimos mapa de color
    vals = manzanas_gdf['est_socio_mean'].dropna()
    if len(vals) > 0:
        vmin, vmax = float(vals.min()), float(vals.max())
    else:
        vmin, vmax = 0.0, 1.0
    colormap = branca.colormap.LinearColormap(
        ['blue','cyan','yellow','orange','red'], vmin=vmin, vmax=vmax
    )
    colormap.caption = "Nivel socioeconómico (est_socio_mean)"
    colormap.add_to(m)

    # --- Función para generar un mapa por cada ubicación ---
    def exterior_coords_list(geom):
        rings = []
        if isinstance(geom, Polygon):
            rings.append(list(geom.exterior.coords))
        elif isinstance(geom, MultiPolygon):
            for poly in geom.geoms:
                rings.append(list(poly.exterior.coords))
            else:
                try:
                    poly = Polygon(geom)
                    rings.append(list(poly.exterior.coords))
                except Exception:
                    pass
            return rings


    # style_fn global para manzanas (reusa colormap)
    def style_fn(feature, cmap=colormap):
        try:
            val = feature['properties'].get('est_socio_mean', None)
            if val is None or pd.isna(val):
                return {'fillOpacity':0.0, 'color':'#999999', 'weight':0.2}
            color = cmap(float(val))
            return {'fillColor': color, 'fillOpacity': 0.7, 'color': color, 'weight': 0.2}
        except Exception:
            return {'fillOpacity':0.0, 'color':'#999999', 'weight':0.2}

    # Creamos un FeatureGroup para los labels (se añadirá AL FINAL para que quede por encima)
    labels_fg = folium.FeatureGroup(name="Isócrona labels", show=True)

    geom_drive = iso_gdf_drive.loc[0, 'geometry']  # shapely geometry

    # 1) Dibujar el contorno del driving (outline)
    folium.GeoJson(
        data = geom_drive.__geo_interface__,
        name = f"Driving {name} (outline)",
        style_function = lambda feat: {'color':'black', 'weight':2, 'fillOpacity':0.0}
    ).add_to(m)

    # 2) Seleccionar manzanas que intersecten la isócrona driving
    manz_inside = manzanas[manzanas.intersects(geom_drive)].copy()

    # Si hay manzanas, pintarlas como heat
    if not manz_inside.empty:
        folium.GeoJson(
            data = manz_inside.to_json(),
            name = f"Manzanas Driving {name} (heat)",
            style_function = style_fn,
            tooltip = folium.GeoJsonTooltip(
                fields=['CVE_AGEB_4','est_socio_mean','POBTOT'],
                aliases=['AGEB','est_socio','POBTOT_manzana'],
                localize=True
            )
        ).add_to(m)

    # Añadimos cluster de servicios dentro del isócrona
    if 'gdf_services' in globals():
        add_services_cluster(m, geom_drive, gdf_services, name=f"{TIPO_DE_DENUE} {name}")

    # Add isochrone polygon
    geom_walk = iso_gdf_walk.loc[0, 'geometry']
    folium.GeoJson(
        geom_walk,
        name=f"{name} (isócrona)",
        style_function=lambda x: {'color':'red','weight':2,'fillOpacity':0.0}
    ).add_to(m)

 # 2) Seleccionamos manzanas que intersecten la isócrona driving
    manz_inside_walk = manzanas[manzanas.intersects(geom_walk)].copy()
    # Si hay manzanas, pintarlas como heat
    if not manz_inside_walk.empty:
        folium.GeoJson(
            data = manz_inside_walk.to_json(),
            name = f"Manzanas Driving {i} (heat)",
            style_function = style_fn,
            tooltip = folium.GeoJsonTooltip(
                fields=['CVE_AGEB_4','est_socio_mean','POBTOT'],
                aliases=['AGEB','est_socio','POBTOT_manzana'],
                localize=True
            )
        ).add_to(m)

    # Añadimos cluster de servicios dentro del isócrona
    if 'gdf_services' in globals():
        add_services_cluster(m, geom_walk, gdf_services, name=f"{TIPO_DE_DENUE} {name}")
    # Add the location marker
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(html=f"""
            <div style="font-size:14px; font-weight:bold; color:black;
                        background-color:white; border-radius:50%;
                        width:24px; height:24px; text-align:center;
                        line-height:24px; border:1px solid black;">
                1
            </div>
        """)
    ).add_to(m)

    # --- Añadimos AL FINAL el FeatureGroup con los labels para que queden por encima ---
    labels_fg.add_to(m)

    folium.LayerControl().add_to(m)
    return m


def count_services_in_iso(iso_gdf):
    geom = iso_gdf.loc[0, 'geometry']
    if 'gdf_services' in globals():
        return int(gdf_services.intersects(geom).sum())
    return 0  # si no existe gdf_services, devuelve 0



# Generamos isócronas individuales
# ------------------------------------------- #
iso_gdfs_walk = []
iso_names_walk = []
iso_gdfs_drive = []
iso_names_drive = []

for i, loc in enumerate(locations, start=1):
    try:
        iso_gdf_walk = build_iso(loc, "foot-walking", 5*60)
        iso_gdfs_walk.append(iso_gdf_walk)
        iso_name_walk = f"Walk {i}"
        iso_names_walk.append(iso_name_walk)

        iso_gdf_drive = build_iso(loc, "driving-car", 5*60)
        iso_gdfs_drive.append(iso_gdf_drive)
        iso_name_drive = f"Drive {i}"
        iso_names_drive.append(iso_name_drive)

        # Generamos mapas individuales
        walk_map = generate_single_location_map(loc, iso_gdf_walk, iso_gdf_walk, iso_name_walk, manzanas)
        drive_map = generate_single_location_map(loc, iso_gdf_walk, iso_gdf_drive, iso_name_drive, manzanas)
        walk_map.save(f'/content/databases/resources/mapa_walk_{i}.html')
        drive_map.save(f'/content/databases/resources/mapa_drive_{i}.html')

        # Convertimos el mapa a imagen
        img_data = walk_map._to_png(5)
        img = Image.open(io.BytesIO(img_data))
        img.save(f'/content/databases/resources/mapa_image_walking{i}.png')
        img.save(os.path.join(resources_path, f'mapa_image_walking{i}.png'))

        img_data = drive_map._to_png(5)
        img = Image.open(io.BytesIO(img_data))
        img.save(f'/content/databases/resources/mapa_image_driving{i}.png')
        img.save(os.path.join(resources_path, f'mapa_image_driving{i}.png'))

        agebs_cnt_walk, _, pop_total_walk, socio_avg_walk = display_info_from_iso(iso_gdf_walk)
        services_cnt_walk = count_services_in_iso(iso_gdf_walk)

        agebs_cnt_drive, _, pop_total_drive, socio_avg_drive = display_info_from_iso(iso_gdf_drive)
        cafes_cnt_drive = count_services_in_iso(iso_gdf_drive)

        counts_walk, props_walk, total_w_walk = socio_class_props_from_manzanas(iso_gdf_walk, manzanas)
        counts_drive, props_drive, total_w_drive = socio_class_props_from_manzanas(iso_gdf_drive, manzanas)

        prop_str_walk = " / ".join([f"{props_walk[f'class_{k}']*100:.1f} %" for k in [1,2,3,4]])
        prop_str_drive = " / ".join([f"{props_drive[f'class_{k}']*100:.1f} %" for k in [1,2,3,4]])

        print(f"{f'Ubicación {i} (Walk)':<18}{agebs_cnt_walk:>8}{services_cnt_walk:>10}{(pop_total_walk or 0):>15}{(socio_avg_walk or 0):>18.2f}   {prop_str_walk}")
        print(f"{f'Ubicación {i} (Drive)':<18}{agebs_cnt_drive:>8}{cafes_cnt_drive:>10}{(pop_total_drive or 0):>15}{(socio_avg_drive or 0):>18.2f}   {prop_str_drive}")

    except Exception as e:
        print(f"{f'Ubicación {i}':<12}{'Error':>8}    {e}")



# Generamos y guardamos mapas interactivos completos
# -------------------------------------------- #
for i, loc_coords in enumerate(locations, start=1):
    try:
        iso_gdf_drive = build_iso(loc_coords, "driving-car", 5*60)
        iso_gdf_walk = build_iso(loc_coords, "foot-walking", 5*60)
        map = generate_single_location_map(loc_coords, iso_gdf_walk, iso_gdf_drive, f"Location {i}", manzanas)
        map.save(f'/content/output/mapas/mapa_interactivo{i}.html')
        map.save(os.path.join(maps_path, f'mapa_interactivo_{i}.html'))


        # Convertimos el mapa a imagen
        img_data = map._to_png(5)
        img = Image.open(io.BytesIO(img_data))
        img.save(f'/content/databases/resources/mapa_image_{i}.png')
        img.save(os.path.join(resources_path, f'mapa_image_{i}.png'))


    except Exception as e:
        print(f"{f'Ubicación {i}':<12}{'Error':>8}    {e}")


# Guardamos info de los puntos en un diccionario
# ------------------------------------------- #
locations_info = []

for i, loc_data in enumerate(locations, start=1):
    try:
        # Encontramos a los isócronas correspondientes basado en su índice
        current_iso_walk = iso_gdfs_walk[i-1]
        current_iso_drive = iso_gdfs_drive[i-1]

        # --- Estadísticas del isócrona driving ---
        agebs_cnt_drive, num_estab_drive, pop_total_drive, socio_avg_drive = display_info_from_iso(current_iso_drive)
        services_cnt_drive = count_services_in_iso(current_iso_drive)

        population = pop_total_drive or 0
        services = services_cnt_drive or 0
        socio_level = socio_avg_drive or 0.0
        personas_por_cafe = population / services if services > 0 else 0.0

        image_path = f"/content/databases/resources/mapa_image_{i}.png"

        locations_info.append({
            "nombre": f"{location_names[i-1]}",
            "poblacion": population,
            "services": services,
            "nivel_socio": socio_level,
            "personas_por_cafe": personas_por_cafe,
            "otras_cosas": 0,
            "imagen": image_path
        })

    except Exception as e:
        print(f"Error procesando la ubicación {i}: {e}")

print("Locations_info construidas:", locations_info)


# Generamos gráficas
# ------------------------------------------------------#

# Gráfica de barras de cantidad de servicios
services_counts_walk = [count_services_in_iso(iso_gdfs_walk[i]) for i in range(len(locations))]
services_counts_drive = [count_services_in_iso(iso_gdfs_drive[i]) for i in range(len(locations))]

x = np.arange(len(location_names))  # Posiciones de las etiquetas
width = 0.35  # Ancho de las barras

fig, ax1 = plt.subplots(figsize=(12, 7))

ax2 = ax1.twinx()
rects1 = ax1.bar(x - width/2, services_counts_walk, width, label='Walking', color='skyblue')
rects2 = ax2.bar(x + width/2, services_counts_drive, width, label='Driving', color='lightcoral')

# Formato
ax1.set_xlabel('Ubicación')
ax1.set_ylabel('Número de servicios (Walking)', color='black')
ax2.set_ylabel('Número de servicios (Driving)', color='black')
ax1.set_title(f'Número de {TIPO_DE_DENUE} por ubicación (Walking vs Driving)')
ax1.set_xticks(x)
ax2.set_xticks(x)
ax1.set_xticklabels(location_names)
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
plt.savefig('/content/databases/resources/cafe_counts_per_location.png')
plt.savefig(os.path.join(resources_path, 'cafe_counts_per_location.png'))

plt.close()



# Gráfica de barras de la población
population_counts_walk = [display_info_from_iso(iso_gdfs_walk[i])[2] for i in range(len(locations))]
population_counts_drive = [display_info_from_iso(iso_gdfs_drive[i])[2] for i in range(len(locations))]
location_names = [f"Location {i+1}" for i in range(len(locations))]

x = np.arange(len(location_names))  # Posiciones de las etiquetas
width = 0.35  # Ancho de las barras

fig, ax1 = plt.subplots(figsize=(12, 7))

# Creamos un segundo eje Y compartiendo el mismo eje X
ax2 = ax1.twinx()

# Barras para población caminando (eje izquierdo - ax1)
rects1 = ax1.bar(x - width/2, population_counts_walk, width,
                 label='Walking', color='blue', alpha=0.6)

# Barras para población en auto (eje derecho - ax2)
rects2 = ax2.bar(x + width/2, population_counts_drive, width,
                 label='Driving', color='red', alpha=0.6)

# Configuramos ejes y etiquetas
ax1.set_xlabel('Ubicación')
ax1.set_ylabel('Población (Walking)', color='Black')
ax2.set_ylabel('Población (Driving)', color='Black')

ax1.set_title('Accesibilidad de Población: Walking vs Driving')
ax1.set_xticks(x)
ax1.set_xticklabels(location_names)

# Cambiamos color de los ticks para que coincidan con los datos
ax1.tick_params(axis='y', labelcolor='Black')
ax2.tick_params(axis='y', labelcolor='Black')

# Añadimos leyendas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()
plt.savefig('/content/databases/resources/population_accessibility.png')
plt.savefig(os.path.join(resources_path, 'population_accessibility.png'))

plt.close()


# Gráficas de pastel
for location_index in range(len(locations)):

    # Obtenemos el isócrona de driving
    iso_gdf_drive = iso_gdfs_drive[location_index]

    # Calculamos las proporciones socioeconómicas de clase para el isócrona
    counts_drive, props_drive, total_w_drive = socio_class_props_from_manzanas(iso_gdf_drive, manzanas)

    # Preparamos datos para el pastel de datos
    labels = [f'Class {k} ({props_drive[f"class_{k}"]*100:.1f}%)' for k in [1, 2, 3, 4] if props_drive[f'class_{k}'] > 0]
    sizes = [props_drive[f'class_{k}'] for k in [1, 2, 3, 4] if props_drive[f'class_{k}'] > 0]
    pie_colors = ['gold', 'yellowgreen', 'lightcoral', 'lightskyblue']

    # Creamos la gráfica de pastel
    plt.figure(figsize=(8, 8))
    plt.pie(sizes, labels=labels, colors=pie_colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal')  # Mismo ratio para que sea un círculo
    plt.title(f'Socioeconomic Level Distribution for Location {location_index + 1} (Driving Isochrone)')
    plt.savefig(f'/content/databases/resources/pie_chart_drive_{location_index + 1}.png')
    plt.savefig(os.path.join(resources_path, f'pie_chart_drive_{location_index + 1}.png'))

    plt.close()

In [ ]:
#@title Generación de Reporte

# --- Configuración de salida ---
output_dir = '/content/output'
output_dir_drive = output_path
output_filename = f'{NOMBRE_REPORTE}.pdf'
output_path = os.path.join(output_dir, output_filename)
output_path_drive = os.path.join(output_dir_drive, output_filename)
os.makedirs(output_dir, exist_ok=True)

# --- Creamos PDF ---
c = canvas.Canvas(output_path, pagesize=letter)

# --- Encabezado global ---
y_pos = 750
c.setFont('Helvetica-Bold', 20)
c.drawString(50, y_pos, NOMBRE_REPORTE)

def flatten_transparent_png(png_path, bg_color=(255, 255, 255)):
    img = Image.open(png_path).convert("RGBA")
    background = Image.new("RGB", img.size, bg_color)  # fondo blanco
    background.paste(img, mask=img.split()[3])  # usar canal alfa como máscara
    new_path = png_path.replace(".png", "_flat.png")
    background.save(new_path, "PNG")
    return new_path

loc_AristorLogo = flatten_transparent_png(LOGO_PNG)
y_pos2 = y_pos

if os.path.exists(loc_AristorLogo):
    try:
        img = ImageReader(loc_AristorLogo)
        img_width, img_height = img.getSize()
        aspect = img_height / float(img_width)
        desired_width = 100
        desired_height = desired_width * aspect
        y_pos2 -= desired_height
        c.drawImage(img, 400, y_pos2, width=desired_width, height=desired_height)
    except Exception as e:
        c.drawString(400, y_pos2, f"Error cargando imagen: {e}")
else:
    c.drawString(50, y_pos2, f"Imagen no encontrada: {loc['imagen']}")


if 'contenido' in globals():
    output_text = "Resumen\n"
    output_text += contenido
else:
    output_text = "No captured output available."


y_pos -= 30

# Añadimos el output al pdf

textobject = c.beginText(50, y_pos) # Ajustamos la posición y debajo del título
textobject.setFont('Times-Roman', 10)

# Calculamos la altura del texto
lines = output_text.splitlines()

line_height = 12  # Línea estimada en puntos

for line in lines:
    textobject.textLine(line)
    y_pos -= line_height  # Actualizamos la posición Y tras cada línea

c.drawText(textobject)

loc_cafe_histogram = '/content/databases/resources/cafe_counts_per_location.png'
if os.path.exists(loc_cafe_histogram):
    try:
        y_pos -= line_height
        img = ImageReader(loc_cafe_histogram)
        img_width, img_height = img.getSize()
        aspect = img_height / float(img_width)
        desired_width = 300
        desired_height = desired_width * aspect
        y_pos -= desired_height
        c.drawImage(img, 50, y_pos, width=desired_width, height=desired_height)
    except Exception as e:
        c.drawString(50, y_pos, f"Error cargando imagen: {e}")
else:
    c.drawString(50, y_pos, f"Imagen no encontrada: {loc_cafe_histogram}")

loc_population_histogram = '/content/databases/resources/population_accessibility.png'
if os.path.exists(loc_population_histogram):
    try:
        y_pos -= line_height
        img = ImageReader(loc_population_histogram)
        img_width, img_height = img.getSize()
        aspect = img_height / float(img_width)
        desired_width = 300
        desired_height = desired_width * aspect
        y_pos -= desired_height
        c.drawImage(img, 50, y_pos, width=desired_width, height=desired_height)
    except Exception as e:
        c.drawString(50, y_pos, f"Error cargando imagen: {e}")
else:
    c.drawString(50, y_pos, f"Imagen no encontrada: {loc_population_histogram}")


# --- Iterar por cada localidad ---
for idx, loc_data in enumerate(locations_info, start=1):
    y_pos = 750
    line_height = 20

    c.showPage()  # Nueva página para cada localidad

    # Encabezado de ubicación
    c.setFont('Helvetica-Bold', 14)
    c.drawString(50, y_pos, f"Ubicación {idx}. {loc_data['nombre']}.")

    y_pos -= line_height

    # Subtítulo
    c.setFont('Times-Roman', 12)
    c.drawString(50, y_pos, "Análisis realizado con parámetros de 5 minutos de manejo y 5 minutos de caminata.")

    y_pos -= line_height

    # Fecha
    fecha_actual = datetime.now().strftime("%d / %m / %Y")
    c.drawString(50, y_pos, fecha_actual)

    datos = [
        ["Población", f"{loc_data['poblacion']:,}"],
        ["Cafés", str(loc_data['services'])],
        ["Nivel Socio Promedio", f"{loc_data['nivel_socio']:.2f}"],
        ["Personas por café", f"{loc_data['personas_por_cafe']:,.1f}"],
        ["Extra", str(loc_data['otras_cosas'])]
    ]

    table = Table(datos)
    table.setStyle(TableStyle([
    ("FONTNAME", (0,0), (-1,-1), "Times-Roman"),
    ("GRID", (0,0), (-1,-1), 1, "black"),
    ]))

    y_pos -= 100
    table.wrapOn(c, 400, 200)   # ajustamos dimensiones
    table.drawOn(c, 50, y_pos)  # posición (x,y)

    # Imagen
    y_pos -= line_height
    c.drawString(50, y_pos, "Mapa General:")

    if os.path.exists(loc_data["imagen"]):
        try:
            y_pos -= line_height
            img = ImageReader(loc_data["imagen"])
            img_width, img_height = img.getSize()
            aspect = img_height / float(img_width)
            desired_width = 500
            desired_height = desired_width * aspect
            y_pos -= desired_height
            c.drawImage(img, 50, y_pos, width=desired_width, height=desired_height)
        except Exception as e:
            c.drawString(50, y_pos, f"Error cargando imagen: {e}")
    else:
        c.drawString(50, y_pos, f"Imagen no encontrada: {loc_data['imagen']}")

    y_pos -= line_height
    y_pos2 = y_pos
    c.drawString(50, y_pos, "Mapa Enfocado en Isócrona Interior:")

    loc_image_walking = f'/content/databases/resources/mapa_image_walking{idx}.png'
    if os.path.exists(loc_image_walking):
        try:
            y_pos -= line_height
            img = ImageReader(loc_image_walking)
            img_width, img_height = img.getSize()
            aspect = img_height / float(img_width)
            desired_width = 250
            desired_height = desired_width * aspect
            y_pos -= desired_height
            c.drawImage(img, 50, y_pos, width=desired_width, height=desired_height)
        except Exception as e:
            c.drawString(50, y_pos, f"Error cargando imagen: {e}")
    else:
        c.drawString(50, y_pos, f"Imagen no encontrada: {loc_image_walking}")

    c.drawString(100 + desired_width, y_pos2, "Grafica de pie Nivel socioeconomico:")
    loc_pieChart = f'/content/databases/resources/pie_chart_drive_{idx}.png'
    if os.path.exists(loc_pieChart):
        try:
            y_pos2 -= line_height
            img = ImageReader(loc_pieChart)
            img_width, img_height = img.getSize()
            aspect = img_height / float(img_width)
            desired_width = 200
            desired_height = desired_width * aspect
            y_pos2 -= desired_height
            c.drawImage(img, 120 + desired_width, y_pos2, width=desired_width, height=desired_height)
        except Exception as e:
            c.drawString(50, y_pos2, f"Error cargando imagen: {e}")
    else:
        c.drawString(50, y_pos2, f"Imagen no encontrada: {loc_pieChart}")

# Guardamos PDF
c.save()
shutil.copy(output_path, output_path_drive)

print(f"PDF creado en: {output_path}")
print(f"PDF también guardado en: {output_path_drive}")